In [1]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf



In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [5]:
df = pd.read_csv("../data/youtube_comments_dataset.csv")

In [6]:
df = df[["Comment", "Predicted_Sentiment"]].dropna()
df["Comment"] = df["Comment"].astype(str)

In [7]:
df = df.drop_duplicates(subset=["Comment"]).reset_index(drop=True)


In [8]:
def clean_text(s):
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " url ", s)
    s = re.sub(r"@\w+", " user ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [9]:
df["text"] = df["Comment"].apply(clean_text)


In [10]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["Predicted_Sentiment"])
class_names = le.classes_
print("Sınıflar:", dict(zip(class_names, range(len(class_names)))))
print("Toplam örnek:", len(df))

Sınıflar: {'Negative': 0, 'Neutral': 1, 'Positive': 2}
Toplam örnek: 23093


In [11]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    df["text"], df["label"], test_size=0.30,
    stratify=df["label"], random_state=SEED)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50,
    stratify=y_tmp, random_state=SEED)


In [12]:
print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

Train: 16165  Val: 3464  Test: 3464


In [13]:
print("\n" + "=" * 60)
print("BASELINE: TF-IDF + Logistic Regression")
print("=" * 60)


BASELINE: TF-IDF + Logistic Regression


In [14]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)
Xtr_tfidf = tfidf.fit_transform(X_train)
Xte_tfidf = tfidf.transform(X_test)

In [15]:
lr = LogisticRegression(max_iter=2000, class_weight="balanced")
lr.fit(Xtr_tfidf, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [16]:
lr_pred = lr.predict(Xte_tfidf)
lr_macro_f1 = f1_score(y_test, lr_pred, average="macro")

In [17]:
print(classification_report(y_test, lr_pred, target_names=class_names, digits=3))
print("Baseline Macro F1:", round(lr_macro_f1, 4))

              precision    recall  f1-score   support

    Negative      0.559     0.727     0.632       637
     Neutral      0.674     0.776     0.721       978
    Positive      0.822     0.671     0.739      1849

    accuracy                          0.711      3464
   macro avg      0.685     0.725     0.697      3464
weighted avg      0.732     0.711     0.714      3464

Baseline Macro F1: 0.6974


In [18]:
VOCAB_SIZE = 20000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)


In [19]:
seqs_train = tokenizer.texts_to_sequences(X_train)
seqs_val   = tokenizer.texts_to_sequences(X_val)
seqs_test  = tokenizer.texts_to_sequences(X_test)


In [20]:
lengths = [len(s) for s in seqs_train]
MAX_LEN = int(np.percentile(lengths, 95))
print("\nMAX_LEN:", MAX_LEN)


MAX_LEN: 78


In [21]:
X_train_pad = pad_sequences(seqs_train, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(seqs_val,   maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(seqs_test,  maxlen=MAX_LEN, padding="post", truncating="post")

In [22]:
y_train_np = y_train.values
y_val_np   = y_val.values
y_test_np  = y_test.values

In [23]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(len(class_names), activation="softmax"),
])

In [24]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
classes = np.unique(y_train_np)
weights = compute_class_weight("balanced", classes=classes, y=y_train_np)
class_weight = dict(zip(classes, weights))
print("Class weights:", {class_names[k]: round(v, 3) for k, v in class_weight.items()})

Class weights: {'Negative': np.float64(1.812), 'Neutral': np.float64(1.181), 'Positive': np.float64(0.624)}


In [26]:
early = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

In [27]:
history = model.fit(
    X_train_pad, y_train_np,
    validation_data=(X_val_pad, y_val_np),
    epochs=10,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early],
    verbose=1,
)

Epoch 1/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 64s 196ms/step - accuracy: 0.5663 - loss: 0.7681 - val_accuracy: 0.7315 - val_loss: 0.6225
Epoch 2/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 77s 175ms/step - accuracy: 0.7936 - loss: 0.4780 - val_accuracy: 0.7919 - val_loss: 0.5154
Epoch 3/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 45s 176ms/step - accuracy: 0.8992 - loss: 0.2686 - val_accuracy: 0.7898 - val_loss: 0.5844
Epoch 4/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 45s 178ms/step - accuracy: 0.9358 - loss: 0.1778 - val_accuracy: 0.7185 - val_loss: 0.8823


In [28]:
print("\n" + "=" * 60)
print("LSTM TEST SONUÇLARI")
print("=" * 60)

y_prob = model.predict(X_test_pad, verbose=0)
lstm_pred = np.argmax(y_prob, axis=1)
lstm_macro_f1 = f1_score(y_test_np, lstm_pred, average="macro")

print(classification_report(y_test_np, lstm_pred, target_names=class_names, digits=3))
print("Confusion matrix (satır=gerçek, sütun=tahmin):")
print(confusion_matrix(y_test_np, lstm_pred))
print("LSTM Macro F1:", round(lstm_macro_f1, 4))


LSTM TEST SONUÇLARI
              precision    recall  f1-score   support

    Negative      0.644     0.700     0.671       637
     Neutral      0.869     0.742     0.801       978
    Positive      0.810     0.849     0.829      1849

    accuracy                          0.791      3464
   macro avg      0.774     0.764     0.767      3464
weighted avg      0.796     0.791     0.792      3464

Confusion matrix (satır=gerçek, sütun=tahmin):
[[ 446   37  154]
 [  39  726  213]
 [ 208   72 1569]]
LSTM Macro F1: 0.7669


In [29]:
print("\n" + "=" * 60)
print("KARŞILAŞTIRMA (Macro F1)")
print("=" * 60)
print(f"Baseline (TF-IDF + LR): {lr_macro_f1:.4f}")
print(f"LSTM                  : {lstm_macro_f1:.4f}")

if lstm_macro_f1 - lr_macro_f1 < 0.01:
    print("Not: LSTM baseline'a belirgin bir üstünlük sağlamadı. "
          "Karmaşıklığın getirisini sorgula.")


KARŞILAŞTIRMA (Macro F1)
Baseline (TF-IDF + LR): 0.6974
LSTM                  : 0.7669


In [30]:
model.save("../model/yt_coms_lstm.keras")